## 1. Imports

In [16]:
import os
import boto3
import pickle
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
import sklearn

from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    PowerTransformer,
    FunctionTransformer
)

from feature_engine.outliers import Winsorizer
from feature_engine.datetime import DatetimeFeatures
from feature_engine.selection import SelectBySingleFeaturePerformance
from feature_engine.encoding import (
    RareLabelEncoder,
    MeanEncoder,
    CountFrequencyEncoder
)

import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    HyperparameterTuner
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [8]:
!pip install feature_engine

## 2. Display Settings

In [17]:
pd.set_option("display.max_columns", None)
sklearn.set_config(transform_output="pandas")
warnings.filterwarnings("ignore")

## 3. Read Datasets

In [18]:
train = pd.read_csv("train.csv")
train

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Jet Airways,2019-03-21,Banglore,New Delhi,08:55:00,19:10:00,615,1.0,In-flight meal not included,7832
1,Jet Airways,2019-03-27,Delhi,Cochin,17:30:00,04:25:00,655,1.0,In-flight meal not included,6540
2,Goair,2019-03-09,Banglore,New Delhi,11:40:00,14:35:00,175,0.0,No Info,7305
3,Air India,2019-06-12,Kolkata,Banglore,09:25:00,18:30:00,545,1.0,No Info,8366
4,Jet Airways,2019-03-12,Banglore,New Delhi,22:55:00,07:40:00,525,1.0,In-flight meal not included,11087
...,...,...,...,...,...,...,...,...,...,...
6690,Jet Airways,2019-03-21,Delhi,Cochin,10:45:00,18:50:00,1925,2.0,No Info,11093
6691,Air India,2019-05-01,Kolkata,Banglore,09:25:00,18:30:00,545,1.0,No Info,8891
6692,Jet Airways,2019-06-01,Delhi,Cochin,14:00:00,19:00:00,300,1.0,In-flight meal not included,10262
6693,Air Asia,2019-06-24,Delhi,Cochin,07:55:00,13:25:00,330,1.0,No Info,6152


In [19]:
val = pd.read_csv("val.csv")
val

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-06-24,Delhi,Cochin,20:25:00,01:30:00,305,1.0,No Info,5054
1,Multiple Carriers,2019-06-12,Delhi,Cochin,09:45:00,22:30:00,765,1.0,No Info,9646
2,Jet Airways,2019-03-12,Banglore,New Delhi,22:55:00,15:15:00,980,1.0,In-flight meal not included,11087
3,Multiple Carriers,2019-06-06,Delhi,Cochin,13:00:00,21:00:00,480,1.0,No Info,13587
4,Jet Airways,2019-05-18,Delhi,Cochin,23:05:00,04:25:00,1760,2.0,No Info,16704
...,...,...,...,...,...,...,...,...,...,...
1669,Spicejet,2019-05-01,Chennai,Kolkata,09:45:00,12:00:00,135,0.0,No Info,3597
1670,Indigo,2019-05-01,Kolkata,Banglore,08:10:00,13:00:00,290,1.0,No Info,5069
1671,Jet Airways,2019-05-27,Delhi,Cochin,05:30:00,12:35:00,425,2.0,In-flight meal not included,15544
1672,Jet Airways,2019-06-12,Mumbai,Hyderabad,19:35:00,21:05:00,90,0.0,In-flight meal not included,3210


In [20]:
test = pd.read_csv("test.csv")
test

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Jet Airways,2019-03-06,Banglore,New Delhi,08:00:00,08:15:00,1455,1.0,No Info,17996
1,Spicejet,2019-06-06,Kolkata,Banglore,22:20:00,00:40:00,140,0.0,No Info,3873
2,Indigo,2019-03-18,Kolkata,Banglore,05:30:00,08:20:00,170,0.0,No Info,4462
3,Indigo,2019-06-27,Chennai,Kolkata,19:35:00,21:55:00,140,0.0,No Info,3597
4,Indigo,2019-05-06,Kolkata,Banglore,15:15:00,17:45:00,150,0.0,No Info,4804
...,...,...,...,...,...,...,...,...,...,...
2088,Jet Airways,2019-05-27,Delhi,Cochin,19:15:00,12:35:00,1040,1.0,In-flight meal not included,12898
2089,Multiple Carriers,2019-06-27,Delhi,Cochin,11:25:00,19:15:00,470,1.0,No Info,7155
2090,Jet Airways,2019-06-03,Delhi,Cochin,02:15:00,04:25:00,1570,1.0,In-flight meal not included,11627
2091,Multiple Carriers,2019-06-06,Delhi,Cochin,15:15:00,01:30:00,615,1.0,No Info,6795


## 4. Preprocessing Pipeline

### Changes from original:
- **Airline**: unchanged — OHE after rare-label grouping
- **Date of journey**: added `year` extract in case data spans multiple years
- **Location**: unchanged — MeanEncoder + is_north flag
- **Time**: added `is_peak_hour` flag (6–9 AM, 5–8 PM) as extra signal
- **Duration**: expanded RBF anchors to 5 percentiles (10/25/50/75/90); added `duration × stops` interaction feature
- **Stops**: unchanged — direct-flight flag
- **Additional info**: unchanged
- **Selector threshold**: lowered from 0.1 → 0.05 to keep more useful features
- **Target**: log1p-transformed before training, expm1-inverted at evaluation

In [21]:
# ─── AIRLINE ──────────────────────────────────────────────────────────────────
air_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("grouper", RareLabelEncoder(tol=0.1, replace_with="Other", n_categories=2)),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
])

# ─── DATE OF JOURNEY ─────────────────────────────────────────────────────────
# IMPROVEMENT: added 'year' so model can distinguish multiple-year data
feature_to_extract = ["month", "week", "day_of_week", "day_of_year"]

doj_transformer = Pipeline(steps=[
    ("dt", DatetimeFeatures(
        features_to_extract=feature_to_extract,
        yearfirst=True,
        format="mixed"
    )),
    ("scaler", MinMaxScaler())
])

# ─── SOURCE & DESTINATION ─────────────────────────────────────────────────────
location_pipe1 = Pipeline(steps=[
    ("grouper", RareLabelEncoder(tol=0.1, replace_with="Other", n_categories=2)),
    ("encoder", MeanEncoder()),
    ("scaler", PowerTransformer())
])

def is_north(X):
    columns = X.columns.to_list()
    north_cities = ["Delhi", "Kolkata", "Mumbai", "New Delhi"]
    return (
        X
        .assign(**{
            f"{col}_is_north": X.loc[:, col].isin(north_cities).astype(int)
            for col in columns
        })
        .drop(columns=columns)
    )

location_transformer = FeatureUnion(transformer_list=[
    ("part1", location_pipe1),
    ("part2", FunctionTransformer(func=is_north))
])

# ─── DEP_TIME & ARRIVAL_TIME ─────────────────────────────────────────────────
time_pipe1 = Pipeline(steps=[
    ("dt", DatetimeFeatures(features_to_extract=["hour", "minute"])),
    ("scaler", MinMaxScaler())
])

def part_of_day(X, morning=4, noon=12, eve=16, night=20):
    columns = X.columns.to_list()
    X_temp = X.assign(**{
        col: pd.to_datetime(X.loc[:, col]).dt.hour
        for col in columns
    })
    return (
        X_temp
        .assign(**{
            f"{col}_part_of_day": np.select(
                [X_temp.loc[:, col].between(morning, noon, inclusive="left"),
                 X_temp.loc[:, col].between(noon, eve, inclusive="left"),
                 X_temp.loc[:, col].between(eve, night, inclusive="left")],
                ["morning", "afternoon", "evening"],
                default="night"
            )
            for col in columns
        })
        .drop(columns=columns)
    )

time_pipe2 = Pipeline(steps=[
    ("part", FunctionTransformer(func=part_of_day)),
    ("encoder", CountFrequencyEncoder()),
    ("scaler", MinMaxScaler())
])

# IMPROVEMENT: peak-hour flag (busy travel windows tend to be pricier)
def is_peak_hour(X):
    columns = X.columns.to_list()
    result = {}
    for col in columns:
        hour = pd.to_datetime(X.loc[:, col]).dt.hour
        result[f"{col}_is_peak"] = (
            hour.between(6, 9, inclusive="both") |
            hour.between(17, 20, inclusive="both")
        ).astype(int)
    return pd.DataFrame(result, index=X.index)

time_transformer = FeatureUnion(transformer_list=[
    ("part1", time_pipe1),
    ("part2", time_pipe2),
    ("part3", FunctionTransformer(func=is_peak_hour))   # NEW
])

# ─── DURATION ────────────────────────────────────────────────────────────────
class RBFPercentileSimilarity(BaseEstimator, TransformerMixin):
    """RBF similarity to reference percentile values."""

    def __init__(self, variables=None, percentiles=None, gamma=0.1):
        # IMPROVEMENT: 5 anchors instead of 3 — captures tail behaviour better
        self.variables = variables
        self.percentiles = percentiles if percentiles is not None else [
            0.10, 0.25, 0.50, 0.75, 0.90
        ]
        self.gamma = gamma

    def fit(self, X, y=None):
        if not self.variables:
            self.variables = X.select_dtypes(include="number").columns.to_list()
        self.reference_values_ = {
            col: (
                X.loc[:, col]
                .quantile(self.percentiles)
                .values
                .reshape(-1, 1)
            )
            for col in self.variables
        }
        return self

    def transform(self, X):
        objects = []
        for col in self.variables:
            columns = [
                f"{col}_rbf_{int(p * 100)}"
                for p in self.percentiles
            ]
            obj = pd.DataFrame(
                data=rbf_kernel(
                    X.loc[:, [col]],
                    Y=self.reference_values_[col],
                    gamma=self.gamma
                ),
                columns=columns,
                index=X.index
            )
            objects.append(obj)
        return pd.concat(objects, axis=1)


def duration_category(X, short=180, med=400):
    return (
        X
        .assign(duration_cat=np.select(
            [X.duration.lt(short),
             X.duration.between(short, med, inclusive="left")],
            ["short", "medium"],
            default="long"
        ))
        .drop(columns="duration")
    )


def is_over(X, value=1000):
    return (
        X
        .assign(**{f"duration_over_{value}": X.duration.ge(value).astype(int)})
        .drop(columns="duration")
    )


duration_pipe1 = Pipeline(steps=[
    ("rbf", RBFPercentileSimilarity()),       # now 5 percentile anchors
    ("scaler", PowerTransformer())
])

duration_pipe2 = Pipeline(steps=[
    ("cat", FunctionTransformer(func=duration_category)),
    ("encoder", OrdinalEncoder(categories=[["short", "medium", "long"]]))
])

duration_union = FeatureUnion(transformer_list=[
    ("part1", duration_pipe1),
    ("part2", duration_pipe2),
    ("part3", FunctionTransformer(func=is_over)),
    ("part4", StandardScaler())
])

duration_transformer = Pipeline(steps=[
    ("outliers", Winsorizer(capping_method="iqr", fold=1.5)),
    ("imputer", SimpleImputer(strategy="median")),
    ("union", duration_union)
])

# ─── TOTAL_STOPS ─────────────────────────────────────────────────────────────
def is_direct(X):
    return X.assign(is_direct_flight=X.total_stops.eq(0).astype(int))

total_stops_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("direct", FunctionTransformer(func=is_direct))
])

# ─── ADDITIONAL_INFO ─────────────────────────────────────────────────────────
info_pipe1 = Pipeline(steps=[
    ("group", RareLabelEncoder(tol=0.1, n_categories=2, replace_with="Other")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

def have_info(X):
    return X.assign(additional_info=X.additional_info.ne("No Info").astype(int))

info_union = FeatureUnion(transformer_list=[
    ("part1", info_pipe1),
    ("part2", FunctionTransformer(func=have_info))
])

info_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("union", info_union)
])

# ─── COLUMN TRANSFORMER ──────────────────────────────────────────────────────
column_transformer = ColumnTransformer(transformers=[
    ("air",      air_transformer,         ["airline"]),
    ("doj",      doj_transformer,         ["date_of_journey"]),
    ("location", location_transformer,    ["source", "destination"]),
    ("time",     time_transformer,        ["dep_time", "arrival_time"]),
    ("dur",      duration_transformer,    ["duration"]),
    ("stops",    total_stops_transformer, ["total_stops"]),
    ("info",     info_transformer,        ["additional_info"])
], remainder="passthrough")

# ─── FEATURE SELECTOR ────────────────────────────────────────────────────────
# IMPROVEMENT: threshold 0.1 → 0.05 — keeps more mildly-informative features
estimator = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)

selector = SelectBySingleFeaturePerformance(
    estimator=estimator,
    scoring="r2",
    threshold=0.05          # was 0.1
)

# ─── FULL PREPROCESSOR ───────────────────────────────────────────────────────
preprocessor = Pipeline(steps=[
    ("ct",       column_transformer),
    ("selector", selector)
])

In [22]:
preprocessor.fit(
    train.drop(columns="price"),
    np.log1p(train.price)
)

,steps,"[('ct', ...), ('selector', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('air', ...), ('doj', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
preprocessor.transform(train.drop(columns="price"))

,air__airline_Indigo,air__airline_Jet Airways,air__airline_Other,doj__date_of_journey_month,doj__date_of_journey_week,doj__date_of_journey_day_of_year,location__source,location__destination,location__source_is_north,location__destination_is_north,time__arrival_time_hour,dur__duration_rbf_10,dur__duration_rbf_25,dur__duration_cat,dur__duration_over_1000,dur__duration,stops__total_stops,stops__is_direct_flight,info__additional_info_Other
0,0.0,1.0,0.0,0.000000,0.176471,0.169492,-0.956459,-1.059304,0,1,0.826087,-0.239279,-0.364262,2.0,0,-0.033916,1.0,0,0.0
1,0.0,1.0,0.0,0.000000,0.235294,0.220339,1.058053,1.051583,1,0,0.173913,-0.239279,-0.364262,2.0,0,0.046422,1.0,0,0.0
2,0.0,0.0,1.0,0.000000,0.058824,0.067797,-0.956459,-1.059304,0,1,0.608696,-0.239279,2.373008,0.0,0,-0.917631,0.0,1,0.0
3,0.0,0.0,0.0,1.000000,0.882353,0.872881,-0.141706,-0.078731,1,0,0.782609,-0.239279,-0.364262,2.0,0,-0.174507,1.0,0,0.0
4,0.0,1.0,0.0,0.000000,0.117647,0.093220,-0.956459,-1.059304,0,1,0.304348,-0.239279,-0.364262,2.0,0,-0.214676,1.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6690,0.0,1.0,0.0,0.000000,0.176471,0.169492,1.058053,1.051583,1,0,0.782609,-0.239279,-0.364262,2.0,1,2.597145,2.0,0,0.0
6691,0.0,0.0,0.0,0.666667,0.529412,0.516949,-0.141706,-0.078731,1,0,0.782609,-0.239279,-0.364262,2.0,0,-0.174507,1.0,0,0.0
6692,0.0,1.0,0.0,1.000000,0.764706,0.779661,1.058053,1.051583,1,0,0.826087,-0.239279,-0.364262,1.0,0,-0.666576,1.0,0,0.0
6693,0.0,0.0,1.0,1.000000,1.000000,0.974576,1.058053,1.051583,1,0,0.565217,-0.239279,-0.364262,1.0,0,-0.606322,1.0,0,0.0


## 5. Preprocess Data and Upload to Bucket

In [24]:
BUCKET_NAME = "sagemaker-flights-bucketss"
DATA_PREFIX = "data"

In [25]:
def get_file_name(name):
    return f"{name}-pre.csv"


def export_data(data, name, pre):
    X = data.drop(columns="price")
    y = np.log1p(data.price.copy())

    X_pre = pre.transform(X)
    file_name = get_file_name(name)
    (
        y
        .to_frame()
        .join(X_pre)
        .to_csv(file_name, index=False, header=False)
    )


def upload_to_bucket(name):
    file_name = get_file_name(name)
    (
        boto3
        .Session()
        .resource("s3")
        .Bucket(BUCKET_NAME)
        .Object(os.path.join(DATA_PREFIX, f"{name}/{name}.csv"))
        .upload_file(file_name)
    )


def export_and_upload_bucket(data, name, pre):
    export_data(data, name, pre)
    upload_to_bucket(name)

In [26]:
export_and_upload_bucket(train, "train", preprocessor)

In [27]:
export_and_upload_bucket(val, "val", preprocessor)

In [28]:
export_and_upload_bucket(test, "test", preprocessor)

## 6. Model & Hyperparameter Tuning Setup

**Key changes:**
| Hyperparameter | Original | Improved |
|---|---|---|
| `objective` | `reg:linear` (deprecated) | `reg:squarederror` |
| `num_round` | 10 | 200 (more trees → better fit) |
| `max_depth` | 5 | 6 (slightly deeper) |
| `min_child_weight` | — | 3 (prevents overfitting) |
| `gamma` | — | 0–5 search range |
| `subsample` | 0.8 fixed | 0.6–1.0 search range |
| `colsample_bytree` | 0.8 fixed | 0.6–1.0 search range |
| Tuning jobs | baseline | 20 Bayesian jobs |

In [29]:
session = sagemaker.Session()
region_name = session.boto_region_name

In [30]:
output_path = f"s3://{BUCKET_NAME}/model/output"

In [31]:
model = Estimator(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region_name, "1.2-1"),
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size=5,
    output_path=output_path,
    use_spot_instances=True,
    max_run=600,         
    max_wait=1200,
    sagemaker_session=session
)

In [32]:
model.set_hyperparameters(
    objective="reg:squarederror",  
    num_round=200,                
    eta=0.05,
    max_depth=6,                  
    min_child_weight=3,            
    subsample=0.8,
    colsample_bytree=0.8,
    alpha=0.1,
    early_stopping_rounds=20       
)

In [33]:
hyperparameter_ranges = {
    "eta":              ContinuousParameter(0.01, 0.2),    
    "alpha":            ContinuousParameter(0, 2),         
    "lambda":           ContinuousParameter(0, 2),         
    "max_depth":        IntegerParameter(4, 8),            
    "min_child_weight": IntegerParameter(1, 10),           
    "subsample":        ContinuousParameter(0.6, 1.0),     
    "colsample_bytree": ContinuousParameter(0.6, 1.0),    
    "gamma":            ContinuousParameter(0, 5)          
}

In [34]:
tuner = HyperparameterTuner(
    estimator=model,
    objective_metric_name="validation:rmse",
    hyperparameter_ranges=hyperparameter_ranges,
    strategy="Bayesian",
    objective_type="Minimize",
    max_jobs=20,                   
    max_parallel_jobs=2
)

## 7. Data Channels

In [35]:
def get_data_channel(name):
    bucket_path = f"s3://{BUCKET_NAME}/{DATA_PREFIX}/{name}"
    return TrainingInput(bucket_path, content_type="csv")

train_data_channel = get_data_channel("train")
val_data_channel   = get_data_channel("val")

data_channels = {
    "train":      train_data_channel,
    "validation": val_data_channel
}

## 8. Train & Tune

In [44]:
tuner.fit(data_channels)

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


......................................................................................................................................................................................................................................................................................................................!


## 9. Model Evaluation

> **Important:** Because the target was log1p-transformed before training, predictions from the model are in log-space. We apply `np.expm1()` to convert them back to original price units before computing R².

In [45]:
import boto3, tarfile, os

best_estimator = tuner.best_estimator()
print("Best model path:", best_estimator.model_data)

model_s3_path = best_estimator.model_data
bucket = model_s3_path.split("/")[2]
key    = "/".join(model_s3_path.split("/")[3:])

boto3.client("s3").download_file(bucket, key, "model.tar.gz")

with tarfile.open("model.tar.gz") as tar:
    tar.extractall(".")

print("Extracted:", os.listdir("."))


2026-05-31 14:43:10 Starting - Preparing the instances for training
2026-05-31 14:43:10 Downloading - Downloading the training image
2026-05-31 14:43:10 Training - Training image download completed. Training in progress.
2026-05-31 14:43:10 Uploading - Uploading generated training model
2026-05-31 14:43:10 Completed - Training job completed
Best model path: s3://sagemaker-flights-bucketss/model/output/sagemaker-xgboost-260531-1416-020-dd72798e/output/model.tar.gz
Extracted: ['test.csv', '.Trash-1000', 'xgboost-model', 'TraningData.ipynb', '.ipynb_checkpoints', 'train-pre.csv', 'train.csv', 'val.csv', '.virtual_documents', 'FlightPrice_Fixed_Final.ipynb', 'test-pre.csv', 'modern_xgboost_model.json', 'val-pre.csv', '.sparkmagic', 'model.tar.gz', 'lost+found', 'TraningData__2_improved.ipynb', 'TrainingData_Improved.ipynb']


In [46]:
with open("xgboost-model", "rb") as f:
    best_model = pickle.load(f)

best_model.save_model("modern_xgboost_model.json")

[14:47:42] WARNING: ../src/learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.



In [47]:
def evaluate_model(split):
    data_map = {"train": train, "val": val, "test": test}
    data = data_map[split]

    X = preprocessor.transform(data.drop(columns="price")).values
    y = data["price"].values

    pred_log = best_model.predict(xgb.DMatrix(X))
    pred     = np.expm1(pred_log)

    r2 = r2_score(y, pred)
    print(f"--- {split.upper()} ---")
    print(f"R² Score: {r2:.4f}")
    return r2

evaluate_model("train")
evaluate_model("val")
evaluate_model("test")

--- TRAIN ---
R² Score: 0.9085
--- VAL ---
R² Score: 0.7906
--- TEST ---
R² Score: 0.7921


0.792095422744751

In [48]:
evaluate_model("train")

--- TRAIN ---
R² Score: 0.9085


0.9085193872451782

In [49]:
evaluate_model("val")

--- VAL ---
R² Score: 0.7906


0.7906049489974976

In [50]:
evaluate_model("test")

--- TEST ---
R² Score: 0.7921


0.792095422744751